# CropCop Track B — 01 Qualification & Protected Claim

This is the **only supported Track-B v5 scientific execution notebook**.

### Qualification
Attach exactly the two immutable datasets produced by Notebook 00:
1. Track-B infrastructure bundle
2. Track-B external bundle

Use **T4 x2**, Internet ON. Configure the Kaggle secret `KAGGLE_API_TOKEN`. Before the multi-hour qualification, the notebook fail-fast verifies the exact T4 x2 surface, installs/verifies the locked Track-B runtime, and then exercises the deterministic Kaggle private publication/individual-file round-trip capability with that locked runtime. The credential is scrubbed before the fresh prediction-blind scientific subprocess and is reloaded only after independent qualification QA to publish the immutable qualification handoff. A release-bound operator hotfix corrects the V1-test guard false positive, applies an exact-equivalent vectorized DINO top-k implementation, and streams scientific-stage progress live so long Kaggle runs are observable.

Set `RUN_MODE = "qualification"`. Do **not** attach a prior qualification dataset in this mode; the notebook rejects it.

### Claim
Attach the **exact same infrastructure and external dataset versions**, plus the exact immutable qualification dataset emitted by the reviewed qualification run.

Set `RUN_MODE = "claim"`, paste the reviewed `qualification_science_sha256`, and keep `KAGGLE_API_TOKEN` configured. The claim path authenticates the qualification bundle and **does not recompute qualification**.

GitHub credentials are never part of Track-B scientific execution. Existing authoritative output directories are never deleted automatically.


In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, time

RUN_MODE = 'qualification'  # qualification | claim
AUTHORIZED_QUALIFICATION_SCIENCE_SHA256 = ''
KAGGLE_OWNER = 'ranamuhammadahmed6'
RELEASE_ID = 'TRACKB_V5_RELEASE_AUTHORITY_v1'
EXPECTED_RUNTIME_SOURCE_COMMIT = 'c31e11da2980d3d41110b1a767e8829cc4306d4c'
NOTEBOOK_STARTED_AT_EPOCH = time.time()
os.environ['TRACKB_NOTEBOOK_STARTED_AT_EPOCH'] = str(NOTEBOOK_STARTED_AT_EPOCH)

if RUN_MODE not in {'qualification', 'claim'}:
    raise RuntimeError('RUN_MODE must be qualification or claim')
if RUN_MODE == 'qualification' and AUTHORIZED_QUALIFICATION_SCIENCE_SHA256:
    raise RuntimeError('Qualification mode must not carry a claim authorization digest')
if RUN_MODE == 'claim':
    value = AUTHORIZED_QUALIFICATION_SCIENCE_SHA256.strip().lower()
    if len(value) != 64 or any(ch not in '0123456789abcdef' for ch in value):
        raise RuntimeError('Claim mode requires the reviewed 64-character qualification science SHA-256')


In [ ]:
INPUT = Path('/kaggle/input')

def _sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def _sha256_json(obj) -> str:
    payload = json.dumps(obj, sort_keys=True, separators=(',', ':'), ensure_ascii=False).encode('utf-8')
    return hashlib.sha256(payload).hexdigest()

def _git_blob_sha1(path: Path) -> str:
    data = path.read_bytes()
    return hashlib.sha1(f'blob {len(data)}\0'.encode('utf-8') + data).hexdigest()

def _single(name: str) -> Path:
    matches = sorted(p.resolve() for p in INPUT.glob(f'**/{name}') if p.is_file())
    if len(matches) != 1:
        raise RuntimeError(f'Expected exactly one {name}; found {matches}')
    return matches[0]

def _role_identity(root: Path):
    root = root.resolve()
    rows = []
    for path in sorted(root.rglob('*')):
        if path.is_symlink():
            raise RuntimeError(f'Attached Track-B role contains a symlink: {path}')
        if not path.is_file():
            continue
        rows.append({
            'path': path.relative_to(root).as_posix(),
            'bytes': int(path.stat().st_size),
            'sha256': _sha256_file(path),
        })
    if not rows:
        raise RuntimeError(f'Attached Track-B role payload is empty: {root}')
    return {
        'file_count': len(rows),
        'total_bytes': sum(int(row['bytes']) for row in rows),
        'content_sha256': _sha256_json(rows),
    }

infra_path = _single('TRACKB_INFRASTRUCTURE_BUNDLE.json')
external_path = _single('TRACKB_EXTERNAL_BUNDLE.json')
infra = json.loads(infra_path.read_text(encoding='utf-8'))
external = json.loads(external_path.read_text(encoding='utf-8'))

if infra.get('bundle_role') != 'TRACKB_INFRASTRUCTURE':
    raise RuntimeError('Attached infrastructure bundle receipt role mismatch')
if external.get('bundle_role') != 'TRACKB_EXTERNAL':
    raise RuntimeError('Attached external bundle receipt role mismatch')

materialization_id = str(infra.get('materialization_id', '')).strip().lower()
if (
    len(materialization_id) != 64
    or any(ch not in '0123456789abcdef' for ch in materialization_id)
):
    raise RuntimeError('Attached Track-B materialization_id is not a 64-character SHA-256')

infra_root = infra_path.parent.resolve()
external_root = external_path.parent.resolve()
if infra_root == external_root:
    raise RuntimeError('Infrastructure and external handoffs must be distinct Kaggle datasets')

qualification_matches = sorted(
    p.resolve() for p in INPUT.glob('**/TRACKB_QUALIFICATION_BUNDLE.json') if p.is_file()
)
if RUN_MODE == 'qualification' and qualification_matches:
    raise RuntimeError(
        'Qualification mode must attach only infrastructure + external handoffs; '
        f'found prior qualification bundle(s): {qualification_matches}'
    )
if RUN_MODE == 'claim' and len(qualification_matches) != 1:
    raise RuntimeError(
        'Claim mode requires exactly one immutable qualification bundle; '
        f'found {qualification_matches}'
    )

if str(infra.get('repository_source_sha', '')) != EXPECTED_RUNTIME_SOURCE_COMMIT:
    raise RuntimeError(
        'Attached infrastructure repository source does not match frozen v5 runtime: '
        f"expected={EXPECTED_RUNTIME_SOURCE_COMMIT}, observed={infra.get('repository_source_sha')}"
    )
if str(external.get('repository_source_sha', '')) != EXPECTED_RUNTIME_SOURCE_COMMIT:
    raise RuntimeError(
        'Attached external repository source does not match frozen v5 runtime'
    )

if str(infra.get('repository_source_sha', '')).lower() != EXPECTED_RUNTIME_SOURCE_COMMIT:
    raise RuntimeError(
        f'Attached Track-B runtime generation mismatch: expected {EXPECTED_RUNTIME_SOURCE_COMMIT}, '
        f"got {infra.get('repository_source_sha')}"
    )

for field in (
    'materialization_id',
    'repository_source_sha',
    'scientific_execution_lock_sha256',
    'scientific_code_attestation_sha256',
    'external_source_lock_sha256',
    'input_materialization_lock_sha256',
    'role_manifest_sha256',
    'role_content_identity',
):
    if infra.get(field) != external.get(field):
        raise RuntimeError(f'Attached Track-B bundles are not paired: mismatch={field}')

roles = {}
for p in sorted(INPUT.glob('**/TRACKB_INPUT_MANIFEST.json')):
    obj = json.loads(p.read_text(encoding='utf-8'))
    role = str(obj.get('role', '')).strip()
    if not role or role in roles:
        raise RuntimeError(f'Invalid/duplicate Track-B role {role!r}: {p}')
    roles[role] = (p.resolve(), obj)
required = {'core', 'historical_compare', 'gvlid_v5', 'irish_potato'}
if set(roles) != required:
    raise RuntimeError(f'Attached Track-B roles mismatch: expected={sorted(required)}, observed={sorted(roles)}')

expected_dataset_root = {
    'core': infra_root,
    'historical_compare': infra_root,
    'gvlid_v5': external_root,
    'irish_potato': external_root,
}
for role, (manifest_path, _obj) in roles.items():
    expected_root = expected_dataset_root[role]
    if expected_root not in manifest_path.parents:
        raise RuntimeError(
            f'Attached Track-B role {role!r} is outside its paired dataset root: '
            f'expected_root={expected_root}, manifest={manifest_path}'
        )

observed_manifest_sha = {role: _sha256_file(path) for role, (path, _) in roles.items()}
if observed_manifest_sha != infra.get('role_manifest_sha256'):
    raise RuntimeError('Attached role-manifest hashes differ from materialization receipt')

# Full multi-gigabyte role-byte verification is intentionally performed once by
# trackb_v4_execute_attached.py before scientific execution. The notebook verifies
# the paired role manifests and executable/code authority first, avoiding a duplicate
# ~17-minute read of the immutable Kaggle inputs.
core_path, core = roles['core']
core_root = core_path.parent.resolve()
files = core.get('files') or {}
for key, receipt_field in (
    ('execution_lock', 'scientific_execution_lock_sha256'),
    ('code_attestation', 'scientific_code_attestation_sha256'),
):
    row = files.get(key)
    if not isinstance(row, dict):
        raise RuntimeError(f'Core manifest lacks {key}')
    artifact = (core_root / str(row.get('path', ''))).resolve()
    if core_root not in artifact.parents or not artifact.is_file():
        raise RuntimeError(f'Unsafe/missing core {key}: {artifact}')
    if _sha256_file(artifact) != infra.get(receipt_field):
        raise RuntimeError(f'Core {key} does not match paired receipt')

repo_rel = str(core.get('repository_root', '')).strip()
REPO = (core_root / repo_rel).resolve()
if core_root not in REPO.parents or not (REPO / 'journal_extension').is_dir():
    raise RuntimeError(f'Embedded repository snapshot missing/unsafe: {REPO}')

for rel, receipt_field in (
    ('journal_extension/track_b_r07/TRACKB_EXTERNAL_SOURCE_LOCK_v2.json', 'external_source_lock_sha256'),
    ('journal_extension/track_b_r07/TRACKB_INPUT_MATERIALIZATION_LOCK_v2.json', 'input_materialization_lock_sha256'),
):
    policy = (REPO / rel).resolve()
    if REPO not in policy.parents or not policy.is_file():
        raise RuntimeError(f'Embedded Track-B policy missing/unsafe: {rel}')
    if _sha256_file(policy) != str(infra.get(receipt_field, '')):
        raise RuntimeError(f'Embedded Track-B policy identity mismatch: {rel}')

attestation_row = files.get('code_attestation')
attestation_path = (core_root / str(attestation_row['path'])).resolve()
attestation = json.loads(attestation_path.read_text(encoding='utf-8'))
attested_paths = set()
for row in attestation.get('files') or []:
    rel = str(row.get('path', '')).replace('\\', '/').strip()
    expected = str(row.get('git_blob_sha1', '')).lower()
    path = (REPO / rel).resolve()
    if REPO not in path.parents or not path.is_file():
        raise RuntimeError(f'Attested embedded file missing/unsafe: {rel}')
    if _git_blob_sha1(path) != expected:
        raise RuntimeError(f'Embedded repository attestation mismatch: {rel}')
    attested_paths.add(rel)

required_executable_paths = {
    'journal_extension/scripts/bootstrap_trackb_runtime.py',
    'journal_extension/scripts/trackb_v4_execute_attached.py',
    'journal_extension/scripts/run_trackb_r07.py',
    'journal_extension/track_b_r07/requirements-trackb.lock.txt',
}
missing_attestation = required_executable_paths - attested_paths
if missing_attestation:
    raise RuntimeError(f'Executable Track-B paths are not code-attested: {sorted(missing_attestation)}')

qualification_manifest = None
if RUN_MODE == 'claim':
    qpath = _single('TRACKB_QUALIFICATION_BUNDLE.json')
    qualification_manifest = json.loads(qpath.read_text(encoding='utf-8'))
    if qualification_manifest.get('role') != 'TRACKB_QUALIFICATION':
        raise RuntimeError('Attached qualification bundle role mismatch')
    if qualification_manifest.get('status') != 'PASS_IMMUTABLE_PREDICTION_BLIND_QUALIFICATION':
        raise RuntimeError('Attached qualification bundle is not terminal PASS')
    if str(qualification_manifest.get('materialization_id', '')) != str(infra['materialization_id']):
        raise RuntimeError('Qualification bundle materialization mismatch')
    if str(qualification_manifest.get('repository_source_sha', '')) != str(infra['repository_source_sha']):
        raise RuntimeError('Qualification bundle repository-source mismatch')
    if str(qualification_manifest.get('external_source_lock_sha256', '')) != str(infra['external_source_lock_sha256']):
        raise RuntimeError('Qualification bundle external-source policy mismatch')
    if str(qualification_manifest.get('input_materialization_lock_sha256', '')) != str(infra['input_materialization_lock_sha256']):
        raise RuntimeError('Qualification bundle materialization-policy mismatch')
    if str(qualification_manifest.get('qualification_science_sha256', '')).lower() != AUTHORIZED_QUALIFICATION_SCIENCE_SHA256.strip().lower():
        raise RuntimeError('Qualification bundle does not match reviewed science SHA')
    evidence_root = (qpath.parent / str(qualification_manifest.get('evidence_root', ''))).resolve()
    if qpath.parent not in evidence_root.parents or not evidence_root.is_dir():
        raise RuntimeError('Qualification evidence root missing/unsafe')
    if _role_identity(evidence_root) != qualification_manifest.get('evidence_content_identity'):
        raise RuntimeError('Qualification evidence bytes differ from published handoff')

print(json.dumps({
    'status': 'PASS_PRECONTROLLER_ATTACHED_AUTHORITY',
    'release_id': RELEASE_ID,
    'expected_runtime_source_commit': EXPECTED_RUNTIME_SOURCE_COMMIT,
    'materialization_id': infra['materialization_id'],
    'repository_source_sha': infra['repository_source_sha'],
    'roles': sorted(roles),
    'claim_qualification_attached': qualification_manifest is not None,
    'full_role_content_verification': 'DEFERRED_TO_CONTROLLER_BEFORE_SCIENCE',
}, indent=2, sort_keys=True))


In [ ]:
WORK = Path('/kaggle/working')
OPERATOR_PREFLIGHT_RECEIPT = WORK / 'TRACKB_V5_OPERATOR_PREFLIGHT.json'

# Hardware is checked before dependency work so a wrong Kaggle accelerator fails immediately.
gpu_probe = subprocess.run(
    ['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'],
    capture_output=True,
    text=True,
    check=False,
    timeout=60,
)
if gpu_probe.returncode != 0:
    raise RuntimeError(
        'T4 x2 preflight could not query nvidia-smi: '
        + (gpu_probe.stderr or gpu_probe.stdout or '')[-1200:]
    )
gpu_names = [line.strip() for line in gpu_probe.stdout.splitlines() if line.strip()]
if len(gpu_names) < 2 or any('T4' not in name.upper() for name in gpu_names[:2]):
    raise RuntimeError(
        f'Track-B v5 qualification requires Kaggle T4 x2; observed GPUs={gpu_names}'
    )

BOOTSTRAP = REPO / 'journal_extension/scripts/bootstrap_trackb_runtime.py'
LOCKFILE = REPO / 'journal_extension/track_b_r07/requirements-trackb.lock.txt'
BOOTSTRAP_RECEIPT = WORK / 'TRACKB_V5_EXECUTION_RUNTIME.json'

subprocess.run([
    sys.executable, str(BOOTSTRAP),
    '--requirements', str(LOCKFILE),
    '--receipt', str(BOOTSTRAP_RECEIPT),
], cwd=REPO, check=True, env=os.environ.copy())

runtime = json.loads(BOOTSTRAP_RECEIPT.read_text(encoding='utf-8'))
if runtime.get('status') != 'PASS':
    raise RuntimeError('Track-B v5 runtime bootstrap failed')
if runtime.get('scientific_execution_requires_fresh_subprocess') is not True:
    raise RuntimeError('Runtime receipt does not require a fresh scientific subprocess')
if runtime['probe'].get('cuda_available') is not True:
    raise RuntimeError('Locked runtime bootstrap does not see CUDA')
runtime_gpu_names = [str(name) for name in runtime['probe'].get('cuda_devices') or []]
if len(runtime_gpu_names) < 2 or any('T4' not in name.upper() for name in runtime_gpu_names[:2]):
    raise RuntimeError(
        f'Locked Track-B runtime does not expose T4 x2; observed={runtime_gpu_names}'
    )

# Exercise publication using the locked Kaggle CLI/runtime, not Kaggle's stock environment.
src_root = str(REPO / 'journal_extension/src')
if src_root not in sys.path:
    sys.path.insert(0, src_root)
from cropcop_je.trackb_r07_ops import (
    configure_runtime_secrets,
    verify_authenticated_kaggle_owner,
    verify_kaggle_publication_capability,
)

operator_preflight = {
    'schema_version': '1.0',
    'status': 'PENDING',
    'materialization_id': materialization_id,
    'expected_kaggle_owner': KAGGLE_OWNER,
    'gpu_names': gpu_names,
    'runtime_gpu_names': runtime_gpu_names,
    'runtime_status': runtime['status'],
    'runtime_versions': runtime['probe']['versions'],
}
try:
    configure_runtime_secrets(require_github=False)
    observed_owner = verify_authenticated_kaggle_owner(KAGGLE_OWNER)
    publication_probe = verify_kaggle_publication_capability(observed_owner)
    if publication_probe.get('status') != 'PASS_KAGGLE_PUBLICATION_CAPABILITY':
        raise RuntimeError(
            f'Kaggle publication capability preflight did not PASS: {publication_probe}'
        )
    publication = publication_probe.get('publication') or {}
    if publication.get('publication_authority') != 'BOUND_MANIFEST_AND_EXACT_BYTE_ROUNDTRIP':
        raise RuntimeError(
            'Kaggle publication probe did not establish exact byte round-trip authority: '
            f'{publication}'
        )
    if int(publication.get('roundtrip_verified_file_count', 0)) < 4:
        raise RuntimeError(
            'Kaggle publication probe did not round-trip all nested multitype payload files'
        )
    operator_preflight.update({
        'status': 'PASS_OPERATOR_PREFLIGHT',
        'authenticated_kaggle_owner': observed_owner,
        'publication_probe_slug': publication_probe['slug'],
        'publication_probe_status': publication_probe['status'],
        'publication_authority': publication.get('publication_authority'),
        'publication_roundtrip_verified_file_count': publication.get(
            'roundtrip_verified_file_count'
        ),
    })
    OPERATOR_PREFLIGHT_RECEIPT.write_text(
        json.dumps(operator_preflight, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
except Exception as exc:
    operator_preflight.update({
        'status': 'FAIL_OPERATOR_PREFLIGHT',
        'error_type': type(exc).__name__,
        'error': str(exc)[-1600:],
    })
    OPERATOR_PREFLIGHT_RECEIPT.write_text(
        json.dumps(operator_preflight, indent=2, sort_keys=True) + '\n',
        encoding='utf-8',
    )
    raise
finally:
    # Publication credentials never enter the fresh prediction-blind subprocess.
    os.environ.pop('KAGGLE_API_TOKEN', None)
    os.environ.pop('CROPCOP_GITHUB_TOKEN', None)

print(json.dumps(operator_preflight, indent=2, sort_keys=True))
print(json.dumps({
    'runtime_status': runtime['status'],
    'cuda_available': runtime['probe']['cuda_available'],
    'cuda_devices': runtime['probe']['cuda_devices'],
    'versions': runtime['probe']['versions'],
}, indent=2, sort_keys=True))


In [ ]:
os.environ.pop('KAGGLE_API_TOKEN', None)
os.environ.pop('CROPCOP_GITHUB_TOKEN', None)

# Release-bound pre-results operator hotfix. It changes only the V1-test safety
# guard's interpretation: explicit "*accessed = false" provenance is allowed,
# while real forbidden test paths/surfaces and non-false access flags remain fatal.
HOTFIX_SHA256 = 'ea582c3a390ae68e15898e9108aeaf2c480d69944094555186f2ee17c8a5da89'
HOTFIX_SOURCE = "from __future__ import annotations\n\nimport os\nimport selectors\nimport subprocess\nimport time\nimport threading\nfrom collections import deque\nfrom typing import Any\n\nHOTFIX_ID = \"TRACKB_V5_V1_GUARD_FALSE_POSITIVE_FIX_v1\"\nACTIVE_ENV = \"TRACKB_V1_GUARD_HOTFIX_ACTIVE\"\n\n\nclass GuardHotfixError(RuntimeError):\n    pass\n\n\ndef _path_is_forbidden(value: str) -> bool:\n    normalized = \"/\" + str(value).replace(\"\\\\\", \"/\").strip().strip(\"/\").lower() + \"/\"\n    if \"/ds-v1-test-consumed/\" in normalized or \"/ds-v1-test/\" in normalized:\n        return True\n    if \"/v1_test/\" in normalized or \"/v1-test/\" in normalized:\n        return True\n    if \"/test_consumed/\" in normalized:\n        return True\n    return False\n\n\ndef _iter_path_values(obj: Any):\n    if isinstance(obj, dict):\n        for child_key, value in obj.items():\n            child_lower = str(child_key).lower()\n            if isinstance(value, str) and (\n                child_lower == \"path\"\n                or child_lower in {\n                    \"data_root\",\n                    \"repository_root\",\n                    \"v1_validation_root\",\n                    \"validation_root\",\n                    \"image_root\",\n                    \"source_root\",\n                }\n                or child_lower.endswith((\"_path\", \"_root\", \"_dir\"))\n            ):\n                yield str(child_key), value\n            else:\n                yield from _iter_path_values(value)\n    elif isinstance(obj, list):\n        for value in obj:\n            yield from _iter_path_values(value)\n\n\ndef _iter_string_values(obj: Any):\n    if isinstance(obj, dict):\n        for value in obj.values():\n            yield from _iter_string_values(value)\n    elif isinstance(obj, list):\n        for value in obj:\n            yield from _iter_string_values(value)\n    elif isinstance(obj, str):\n        yield obj\n\n\ndef _iter_access_flags(obj: Any):\n    if isinstance(obj, dict):\n        for child_key, value in obj.items():\n            child_lower = str(child_key).lower()\n            if child_lower in {\n                \"v1_test_accessed\",\n                \"v1_test_image_bytes_accessed\",\n                \"consumed_v1_test_accessed\",\n            }:\n                yield str(child_key), value\n            yield from _iter_access_flags(value)\n    elif isinstance(obj, list):\n        for value in obj:\n            yield from _iter_access_flags(value)\n\n\ndef validate_manifest_safety(role: str, manifest: dict[str, Any]) -> None:\n    if not isinstance(manifest, dict):\n        raise GuardHotfixError(f\"Track-B input manifest is not an object: role={role!r}\")\n\n    for value in _iter_string_values(manifest):\n        if value.strip().upper() == \"DS-V1-TEST-CONSUMED\":\n            raise GuardHotfixError(\n                f\"forbidden consumed V1-test surface referenced by input role {role!r}\"\n            )\n\n    for key, value in _iter_access_flags(manifest):\n        if value is not False:\n            raise GuardHotfixError(\n                f\"consumed V1-test access flag is not explicitly false: role={role!r}, \"\n                f\"field={key!r}, value={value!r}\"\n            )\n\n    for key, value in _iter_path_values(manifest):\n        if _path_is_forbidden(value):\n            raise GuardHotfixError(\n                f\"forbidden consumed V1-test path referenced by input role {role!r}: \"\n                f\"field={key!r}, value={value!r}\"\n            )\n\n    if role == \"historical_compare\":\n        if manifest.get(\"coverage_scope\") != \"V1_TRAIN_VAL_ONLY\":\n            raise GuardHotfixError(\n                \"historical_compare must remain on the V1_TRAIN_VAL_ONLY safe surface\"\n            )\n        if manifest.get(\"v1_test_image_bytes_accessed\") is not False:\n            raise GuardHotfixError(\n                \"historical_compare must explicitly assert v1_test_image_bytes_accessed=false\"\n            )\n        if manifest.get(\"ext_i_eligible\") is not False:\n            raise GuardHotfixError(\n                \"historical_compare must remain ineligible for EXT-I under the safe route\"\n            )\n        if manifest.get(\"maximum_evidence_grade\") != \"EXT-S\":\n            raise GuardHotfixError(\n                \"historical_compare safe route must remain capped at EXT-S\"\n            )\n\n\ndef _optimized_topk_cosine_neighbors(\n    query_features,\n    reference_features,\n    *,\n    k: int = 50,\n    device=\"cuda\",\n    block_rows: int = 256,\n):\n    \"\"\"Exact-equivalent deterministic cosine top-k with vectorized tie detection.\"\"\"\n    import numpy as np\n    import torch\n    from cropcop_je.trackb_r07 import TrackBError\n\n    q = np.asarray(query_features, dtype=np.float32)\n    r = np.asarray(reference_features, dtype=np.float32)\n    if q.ndim != 2 or r.ndim != 2 or q.shape[1] != r.shape[1]:\n        raise TrackBError(\"feature matrix dimensionality mismatch\")\n    if len(r) < k:\n        raise TrackBError(f\"reference feature surface has fewer than top-k={k} rows\")\n\n    ref = torch.from_numpy(r).to(device)\n    out_idx = np.empty((len(q), k), dtype=np.int64)\n    out_score = np.empty((len(q), k), dtype=np.float32)\n\n    with torch.no_grad():\n        for start in range(0, len(q), int(block_rows)):\n            block = torch.from_numpy(q[start:start + int(block_rows)]).to(device)\n            score = block @ ref.T\n            values, indices = torch.topk(score, k=k, dim=1, largest=True, sorted=True)\n\n            threshold = values[:, -1:].clone()\n            strict_counts = (score > threshold).sum(dim=1)\n            equal_counts = (score == threshold).sum(dim=1)\n            ambiguous = (strict_counts + equal_counts) > int(k)\n\n            index_order = torch.argsort(indices, dim=1, stable=True)\n            canonical_idx = torch.gather(indices, 1, index_order)\n            canonical_values = torch.gather(values, 1, index_order)\n            score_order = torch.argsort(\n                canonical_values, dim=1, descending=True, stable=True\n            )\n            indices = torch.gather(canonical_idx, 1, score_order)\n            values = torch.gather(canonical_values, 1, score_order)\n\n            for row_index in torch.nonzero(\n                ambiguous, as_tuple=False\n            ).flatten().tolist():\n                row_threshold = threshold[row_index, 0]\n                strict_idx = torch.nonzero(\n                    score[row_index] > row_threshold, as_tuple=False\n                ).flatten()\n                tie_idx = torch.nonzero(\n                    score[row_index] == row_threshold, as_tuple=False\n                ).flatten()\n                slots = int(k) - int(strict_idx.numel())\n                if slots < 0:\n                    raise TrackBError(\"top-k cutoff accounting became inconsistent\")\n                chosen_ties = torch.sort(tie_idx).values[:slots]\n                chosen = torch.cat((strict_idx, chosen_ties), dim=0)\n                if int(chosen.numel()) != int(k):\n                    raise TrackBError(\n                        \"deterministic top-k tie resolution did not produce k neighbors\"\n                    )\n                chosen_scores = score[row_index, chosen]\n                order = torch.argsort(\n                    chosen_scores, descending=True, stable=True\n                )\n                indices[row_index] = chosen[order]\n                values[row_index] = chosen_scores[order]\n\n            out_idx[start:start + len(block)] = indices.cpu().numpy()\n            out_score[start:start + len(block)] = values.cpu().numpy()\n\n            completed = min(start + len(block), len(q))\n            if completed == len(q) or completed % max(int(block_rows) * 16, 1) == 0:\n                print(\n                    f\"DINO top-k {completed:,}/{len(q):,} queries against \"\n                    f\"{len(r):,} references\",\n                    flush=True,\n                )\n    return out_idx, out_score\n\n\n_ORIGINAL_VERIFY_ORB_PAIR = None\n_ORB_MATCHER_LOCAL = threading.local()\n\n\ndef _optimized_verify_orb_pair(\n    a,\n    b,\n    *,\n    policy,\n    rng_seed: int = 0,\n):\n    \"\"\"Exact verifier with one stateless BFMatcher reused per worker thread.\"\"\"\n    import cv2\n    import numpy as np\n    from cropcop_je.trackb_r07_audit import _CV2_RANSAC_LOCK, _coverage\n\n    desc_a, desc_b = a[\"desc\"], b[\"desc\"]\n    kp_a, kp_b = a[\"xy\"], b[\"xy\"]\n    result = {\n        \"accepted\": False,\n        \"decision_stage\": \"UNRESOLVED\",\n        \"good_matches\": 0,\n        \"normalized_good_match_ratio\": 0.0,\n        \"homography_inliers\": 0,\n        \"inlier_ratio\": 0.0,\n        \"coverage_a\": 0.0,\n        \"coverage_b\": 0.0,\n        \"median_symmetric_reprojection_px\": None,\n    }\n    if (\n        len(desc_a) < policy.minimum_good_matches\n        or len(desc_b) < policy.minimum_good_matches\n        or len(kp_a) < policy.minimum_good_matches\n        or len(kp_b) < policy.minimum_good_matches\n    ):\n        result[\"decision_stage\"] = \"INSUFFICIENT_DESCRIPTORS\"\n        return result\n\n    matcher = getattr(_ORB_MATCHER_LOCAL, \"matcher\", None)\n    if matcher is None:\n        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)\n        _ORB_MATCHER_LOCAL.matcher = matcher\n    pairs = matcher.knnMatch(desc_a, desc_b, k=2)\n    good = [m for m, n in pairs if m.distance < policy.lowe_ratio * n.distance]\n    result[\"good_matches\"] = len(good)\n    ratio = len(good) / max(min(len(kp_a), len(kp_b)), 1)\n    result[\"normalized_good_match_ratio\"] = ratio\n    if (\n        len(good) < policy.minimum_good_matches\n        or ratio < policy.minimum_normalized_good_match_ratio\n    ):\n        result[\"decision_stage\"] = \"GOOD_MATCH_GATE\"\n        return result\n\n    pts_a = np.float32([kp_a[m.queryIdx] for m in good])\n    pts_b = np.float32([kp_b[m.trainIdx] for m in good])\n    with _CV2_RANSAC_LOCK:\n        cv2.setRNGSeed(int(rng_seed) & 0x7FFFFFFF)\n        H, mask = cv2.findHomography(\n            pts_a,\n            pts_b,\n            cv2.RANSAC,\n            policy.homography_ransac_reprojection_px,\n        )\n    if H is None or mask is None:\n        result[\"decision_stage\"] = \"HOMOGRAPHY_FAIL\"\n        return result\n\n    mask = mask.reshape(-1).astype(bool)\n    inliers = int(mask.sum())\n    inlier_ratio = inliers / len(good)\n    result[\"homography_inliers\"] = inliers\n    result[\"inlier_ratio\"] = inlier_ratio\n    if (\n        inliers < policy.minimum_homography_inliers\n        or inlier_ratio < policy.minimum_inlier_ratio\n    ):\n        result[\"decision_stage\"] = \"INLIER_GATE\"\n        return result\n\n    in_a, in_b = pts_a[mask], pts_b[mask]\n    cov_a, cov_b = _coverage(in_a, a[\"shape\"]), _coverage(in_b, b[\"shape\"])\n    result[\"coverage_a\"], result[\"coverage_b\"] = cov_a, cov_b\n    if (\n        cov_a < policy.minimum_convex_hull_coverage_each_image\n        or cov_b < policy.minimum_convex_hull_coverage_each_image\n    ):\n        result[\"decision_stage\"] = \"COVERAGE_GATE\"\n        return result\n\n    try:\n        H_inv = np.linalg.inv(H)\n    except np.linalg.LinAlgError:\n        result[\"decision_stage\"] = \"HOMOGRAPHY_INVERSE_FAIL\"\n        return result\n\n    fwd = cv2.perspectiveTransform(in_a.reshape(-1, 1, 2), H).reshape(-1, 2)\n    rev = cv2.perspectiveTransform(in_b.reshape(-1, 1, 2), H_inv).reshape(-1, 2)\n    symmetric = 0.5 * (\n        np.linalg.norm(fwd - in_b, axis=1)\n        + np.linalg.norm(rev - in_a, axis=1)\n    )\n    median = float(np.median(symmetric))\n    result[\"median_symmetric_reprojection_px\"] = median\n    result[\"accepted\"] = bool(\n        median <= policy.maximum_median_symmetric_reprojection_px\n    )\n    result[\"decision_stage\"] = (\n        \"ACCEPT\" if result[\"accepted\"] else \"REPROJECTION_GATE\"\n    )\n    return result\n\n\ndef selftest_orb_verifier_equivalence() -> dict[str, object]:\n    \"\"\"Prove BFMatcher reuse preserves the frozen verifier exactly.\"\"\"\n    import types\n    import cv2\n    import numpy as np\n\n    if _ORIGINAL_VERIFY_ORB_PAIR is None:\n        raise GuardHotfixError(\"original ORB verifier is unavailable for equivalence QA\")\n\n    policy = types.SimpleNamespace(\n        lowe_ratio=0.75,\n        minimum_good_matches=20,\n        minimum_normalized_good_match_ratio=0.12,\n        homography_ransac_reprojection_px=5.0,\n        minimum_homography_inliers=12,\n        minimum_inlier_ratio=0.35,\n        minimum_convex_hull_coverage_each_image=0.1,\n        maximum_median_symmetric_reprojection_px=3.0,\n    )\n    rng = np.random.default_rng(1907)\n\n    # Case 1: descriptor-count rejection.\n    small_desc = rng.integers(0, 256, size=(8, 32), dtype=np.uint8)\n    small_xy = rng.uniform(0, 255, size=(8, 2)).astype(np.float32)\n    cases = [\n        (\n            \"insufficient\",\n            {\"desc\": small_desc, \"xy\": small_xy, \"shape\": (256, 256)},\n            {\"desc\": small_desc.copy(), \"xy\": small_xy.copy(), \"shape\": (256, 256)},\n            11,\n        )\n    ]\n\n    # Case 2: normal random pair expected to fail the Lowe/good-match gate.\n    random_a = rng.integers(0, 256, size=(96, 32), dtype=np.uint8)\n    random_b = rng.integers(0, 256, size=(104, 32), dtype=np.uint8)\n    cases.append(\n        (\n            \"random_gate\",\n            {\n                \"desc\": random_a,\n                \"xy\": rng.uniform(0, 255, size=(96, 2)).astype(np.float32),\n                \"shape\": (256, 256),\n            },\n            {\n                \"desc\": random_b,\n                \"xy\": rng.uniform(0, 255, size=(104, 2)).astype(np.float32),\n                \"shape\": (256, 256),\n            },\n            17,\n        )\n    )\n\n    # Case 3: a deterministic translated surface that should traverse homography,\n    # coverage, and reprojection logic.\n    side = 8\n    xs, ys = np.meshgrid(\n        np.linspace(20, 220, side, dtype=np.float32),\n        np.linspace(20, 220, side, dtype=np.float32),\n    )\n    xy_a = np.stack((xs.reshape(-1), ys.reshape(-1)), axis=1)\n    xy_b = xy_a + np.asarray([7.0, 5.0], dtype=np.float32)\n    desc = rng.integers(0, 256, size=(len(xy_a), 32), dtype=np.uint8)\n    cases.append(\n        (\n            \"translated_accept\",\n            {\"desc\": desc, \"xy\": xy_a, \"shape\": (256, 256)},\n            {\"desc\": desc.copy(), \"xy\": xy_b, \"shape\": (256, 256)},\n            23,\n        )\n    )\n\n    for name, a, b, seed in cases:\n        expected = _ORIGINAL_VERIFY_ORB_PAIR(\n            a,\n            b,\n            policy=policy,\n            rng_seed=seed,\n        )\n        observed = _optimized_verify_orb_pair(\n            a,\n            b,\n            policy=policy,\n            rng_seed=seed,\n        )\n        if expected != observed:\n            raise GuardHotfixError(\n                f\"ORB verifier equivalence mismatch for {name}: \"\n                f\"expected={expected!r}, observed={observed!r}\"\n            )\n\n    return {\n        \"status\": \"PASS_ORB_VERIFIER_EXACT_EQUIVALENCE\",\n        \"opencv_version\": str(cv2.__version__),\n        \"cases\": [name for name, *_rest in cases],\n        \"matcher_reuse\": \"THREAD_LOCAL_STATELESS_BFMATCHER\",\n    }\n\n\ndef selftest_dino_topk_equivalence() -> dict[str, object]:\n    \"\"\"Prove optimized top-k equals the original reference under the active Torch runtime.\"\"\"\n    import numpy as np\n    import torch\n    from cropcop_je.trackb_r07 import TrackBError\n\n    def reference(q, r, *, k):\n        q = np.asarray(q, dtype=np.float32)\n        r = np.asarray(r, dtype=np.float32)\n        ref = torch.from_numpy(r)\n        out_idx = np.empty((len(q), k), dtype=np.int64)\n        out_score = np.empty((len(q), k), dtype=np.float32)\n        with torch.no_grad():\n            score = torch.from_numpy(q) @ ref.T\n            values, indices = torch.topk(score, k=k, dim=1, largest=True, sorted=True)\n            for row_index in range(len(q)):\n                threshold = values[row_index, -1]\n                strict_idx = torch.nonzero(\n                    score[row_index] > threshold, as_tuple=False\n                ).flatten()\n                tie_idx = torch.nonzero(\n                    score[row_index] == threshold, as_tuple=False\n                ).flatten()\n                slots = int(k) - int(strict_idx.numel())\n                if slots < 0:\n                    raise TrackBError(\"reference top-k cutoff accounting became inconsistent\")\n                chosen_ties = torch.sort(tie_idx).values[:slots]\n                chosen = torch.cat((strict_idx, chosen_ties), dim=0)\n                chosen_scores = score[row_index, chosen]\n                order = torch.argsort(chosen_scores, descending=True, stable=True)\n                out_idx[row_index] = chosen[order].cpu().numpy()\n                out_score[row_index] = chosen_scores[order].cpu().numpy()\n        return out_idx, out_score\n\n    rng = np.random.default_rng(1701)\n    q = rng.normal(size=(37, 23)).astype(np.float32)\n    r = rng.normal(size=(113, 23)).astype(np.float32)\n    q /= np.linalg.norm(q, axis=1, keepdims=True)\n    r /= np.linalg.norm(r, axis=1, keepdims=True)\n\n    cases = [\n        (\"random\", q, r, 11),\n        (\n            \"cutoff_ties\",\n            np.asarray([[1.0, 0.0], [0.0, 1.0]], dtype=np.float32),\n            np.asarray([\n                [1.0, 0.0],\n                [1.0, 0.0],\n                [1.0, 0.0],\n                [0.5, 0.5],\n                [0.5, 0.5],\n                [0.0, 1.0],\n                [0.0, 1.0],\n            ], dtype=np.float32),\n            2,\n        ),\n    ]\n\n    for name, case_q, case_r, k in cases:\n        expected_idx, expected_score = reference(case_q, case_r, k=k)\n        observed_idx, observed_score = _optimized_topk_cosine_neighbors(\n            case_q, case_r, k=k, device=\"cpu\", block_rows=8\n        )\n        if not np.array_equal(expected_idx, observed_idx):\n            raise GuardHotfixError(f\"DINO top-k equivalence index mismatch: {name}\")\n        if not np.array_equal(expected_score, observed_score):\n            raise GuardHotfixError(f\"DINO top-k equivalence score mismatch: {name}\")\n\n    return {\n        \"status\": \"PASS_DINO_TOPK_EXACT_EQUIVALENCE\",\n        \"torch_version\": str(torch.__version__),\n        \"cases\": [name for name, *_rest in cases],\n    }\n\n\ndef _observable_encode_audit_features(\n    model,\n    image_paths,\n    transform,\n    device,\n    *,\n    batch_size: int = 64,\n):\n    \"\"\"Exact-equivalent DINO encoding with bounded progress logging.\"\"\"\n    import numpy as np\n    import torch\n    from cropcop_je.trackb_r07_audit import _canonical_rgb\n\n    model = model.to(device).eval()\n    outputs = []\n    with torch.no_grad():\n        for start in range(0, len(image_paths), int(batch_size)):\n            batch_paths = image_paths[start:start + int(batch_size)]\n            tensors = [transform(_canonical_rgb(path)) for path in batch_paths]\n            x = torch.stack(tensors, dim=0).to(device, non_blocking=True)\n            features = model.forward_features(x)\n            features = model.forward_head(features, pre_logits=True)\n            features = torch.nn.functional.normalize(features.float(), p=2, dim=1)\n            outputs.append(features.cpu().numpy().astype(np.float32, copy=False))\n            completed = min(start + len(batch_paths), len(image_paths))\n            if completed == len(image_paths) or completed % 2048 < len(batch_paths):\n                print(\n                    f\"DINO feature encoding {completed:,}/{len(image_paths):,}\",\n                    flush=True,\n                )\n    if not outputs:\n        return np.empty((0, 0), dtype=np.float32)\n    return np.concatenate(outputs, axis=0)\n\n\ndef _stream_science_command(original_run_checked, ops_module):\n    def run_checked(args, *, cwd=None, timeout=3600):\n        is_science_runner = any(\n            str(part).endswith(\"/run_trackb_r07.py\")\n            or str(part).endswith(\"\\\\run_trackb_r07.py\")\n            for part in args\n        )\n        if not is_science_runner:\n            return original_run_checked(args, cwd=cwd, timeout=timeout)\n\n        env = dict(os.environ)\n        env[\"GIT_TERMINAL_PROMPT\"] = \"0\"\n        started = time.monotonic()\n        tail = deque(maxlen=200)\n        proc = subprocess.Popen(\n            args,\n            cwd=None if cwd is None else str(cwd),\n            env=env,\n            stdout=subprocess.PIPE,\n            stderr=subprocess.STDOUT,\n            text=True,\n            bufsize=1,\n        )\n        selector = selectors.DefaultSelector()\n        try:\n            assert proc.stdout is not None\n            selector.register(proc.stdout, selectors.EVENT_READ)\n            while True:\n                elapsed = time.monotonic() - started\n                remaining = float(timeout) - elapsed\n                if remaining <= 0:\n                    proc.terminate()\n                    try:\n                        proc.wait(timeout=20)\n                    except subprocess.TimeoutExpired:\n                        proc.kill()\n                        proc.wait(timeout=20)\n                    raise subprocess.TimeoutExpired(args, timeout)\n\n                events = selector.select(timeout=min(1.0, remaining))\n                if events:\n                    line = proc.stdout.readline()\n                    if line:\n                        safe = ops_module.redact(line.rstrip(\"\\n\"))\n                        tail.append(safe)\n                        print(safe, flush=True)\n\n                if proc.poll() is not None:\n                    # Drain any complete lines already buffered at process exit.\n                    for line in proc.stdout:\n                        safe = ops_module.redact(line.rstrip(\"\\n\"))\n                        tail.append(safe)\n                        print(safe, flush=True)\n                    break\n            rc = proc.wait()\n        finally:\n            selector.close()\n            if proc.stdout is not None:\n                proc.stdout.close()\n\n        if rc != 0:\n            detail = \"\\n\".join(tail)\n            if len(detail) > 12000:\n                detail = detail[-12000:]\n            raise ops_module.TrackBOpsError(\n                f\"command failed rc={rc}: {' '.join(map(str, args))}\\n{detail}\"\n            )\n        return subprocess.CompletedProcess(args, rc, stdout=\"\", stderr=\"\")\n\n    return run_checked\n\n\ndef install(expected_source_sha256: str) -> dict[str, str]:\n    expected_source_sha256 = str(expected_source_sha256).strip().lower()\n    if len(expected_source_sha256) != 64 or any(\n        ch not in \"0123456789abcdef\" for ch in expected_source_sha256\n    ):\n        raise GuardHotfixError(\"operator hotfix source SHA-256 is missing or malformed\")\n\n    from cropcop_je import trackb_r07\n    from cropcop_je import trackb_r07_audit\n    from cropcop_je import trackb_r07_ops\n\n    def _strict_no_v1_test_surface(inputs):\n        for bundle in inputs.values():\n            validate_manifest_safety(str(bundle.role), bundle.manifest)\n\n    global _ORIGINAL_VERIFY_ORB_PAIR\n    _ORIGINAL_VERIFY_ORB_PAIR = trackb_r07_audit.verify_orb_pair\n\n    trackb_r07.assert_no_v1_test_surface = _strict_no_v1_test_surface\n    trackb_r07_audit.topk_cosine_neighbors = _optimized_topk_cosine_neighbors\n    trackb_r07_audit.encode_audit_features = _observable_encode_audit_features\n    trackb_r07_audit.verify_orb_pair = _optimized_verify_orb_pair\n\n    original_run_checked = trackb_r07_ops.run_checked\n    trackb_r07_ops.run_checked = _stream_science_command(\n        original_run_checked, trackb_r07_ops\n    )\n\n    os.environ[ACTIVE_ENV] = expected_source_sha256\n    return {\n        \"status\": \"PASS_TRACKB_OPERATOR_HOTFIX_INSTALLED\",\n        \"dino_topk_optimization\": \"EXACT_EQUIVALENT_VECTORIZED_CUTOFF_TIE_DETECTION\",\n        \"dino_feature_progress_logging\": True,\n        \"orb_matcher_reuse\": \"THREAD_LOCAL_STATELESS_BFMATCHER\",\n        \"science_subprocess_live_streaming\": True,\n        \"hotfix_id\": HOTFIX_ID,\n        \"source_sha256\": expected_source_sha256,\n    }\n"
HOTFIX_ROOT = WORK / 'trackb_v5_operator_hotfix'
HOTFIX_ROOT.mkdir(parents=True, exist_ok=True)
HOTFIX_PATH = HOTFIX_ROOT / 'trackb_v5_v1_guard_hotfix.py'
HOTFIX_PATH.write_text(HOTFIX_SOURCE, encoding='utf-8')
if _sha256_file(HOTFIX_PATH) != HOTFIX_SHA256:
    raise RuntimeError('Track-B operator hotfix source hash mismatch')
(HOTFIX_ROOT / 'sitecustomize.py').write_text(
    "from trackb_v5_v1_guard_hotfix import install\n"
    f"install('{HOTFIX_SHA256}')\n",
    encoding='utf-8',
)

controller_env = os.environ.copy()
existing_pythonpath = controller_env.get('PYTHONPATH', '')
controller_env['PYTHONPATH'] = os.pathsep.join(
    [
        str(HOTFIX_ROOT),
        str(REPO / 'journal_extension/src'),
        existing_pythonpath,
    ]
)

# Seconds-long compatibility smoke before the controller performs the one expensive
# full-byte verification pass. This catches guard/manifest drift immediately.
discovery_smoke = subprocess.run(
    [
        sys.executable,
        '-c',
        (
            "import json, os; "
            "from cropcop_je.trackb_r07 import discover_kaggle_inputs; "
            "from trackb_v5_v1_guard_hotfix import selftest_dino_topk_equivalence, selftest_orb_verifier_equivalence; "
            "eq=selftest_dino_topk_equivalence(); orb_eq=selftest_orb_verifier_equivalence(); "
            "found=discover_kaggle_inputs('/kaggle/input'); "
            "assert os.environ.get('TRACKB_V1_GUARD_HOTFIX_ACTIVE') == 'ea582c3a390ae68e15898e9108aeaf2c480d69944094555186f2ee17c8a5da89'; "
            "print(json.dumps({'status':'PASS_FAST_INPUT_DISCOVERY_SMOKE',"
            "'roles':sorted(found),'hotfix_active':True,"
            "'dino_topk_equivalence':eq,'orb_verifier_equivalence':orb_eq},sort_keys=True))"
        ),
    ],
    capture_output=True,
    text=True,
    check=False,
    timeout=120,
    env=controller_env,
)
if discovery_smoke.returncode != 0:
    raise RuntimeError(
        'Track-B fast input-discovery smoke failed before expensive verification:\n'
        + (discovery_smoke.stderr or discovery_smoke.stdout or '')[-4000:]
    )
print(discovery_smoke.stdout.strip())

CONTROLLER = REPO / 'journal_extension/scripts/trackb_v4_execute_attached.py'
OUT = WORK / 'trackb_r07'
SCRATCH = Path('/kaggle/tmp/cropcop_trackb_r07_v5')

if OUT.exists() and any(OUT.iterdir()):
    raise RuntimeError(f'Refusing to delete/overwrite existing authoritative Track-B output: {OUT}')
OUT.mkdir(parents=True, exist_ok=True)
if SCRATCH.exists():
    shutil.rmtree(SCRATCH)

cmd = [
    sys.executable, str(CONTROLLER),
    '--input-root', '/kaggle/input',
    '--output-root', str(OUT),
    '--scratch-root', str(SCRATCH),
    '--device', 'cuda:0',
    '--mode', RUN_MODE,
    '--kaggle-owner', KAGGLE_OWNER,
]
if RUN_MODE == 'claim':
    cmd += [
        '--authorized-qualification-science-sha256',
        AUTHORIZED_QUALIFICATION_SCIENCE_SHA256.strip().lower(),
    ]

subprocess.run(cmd, cwd=REPO, check=True, env=controller_env)


In [ ]:
receipt_path = Path('/kaggle/working/trackb_r07') / 'TRACKB_V5_EXECUTION_RECEIPT.json'
receipt = json.loads(receipt_path.read_text(encoding='utf-8'))

if RUN_MODE == 'qualification':
    if receipt.get('status') != 'PASS_TRACKB_PREINFERENCE_QUALIFICATION':
        raise RuntimeError(f"Qualification did not PASS: {receipt.get('status')}")
    if receipt.get('protected_external_prediction_count') != 0:
        raise RuntimeError('Qualification produced protected external predictions')
    if receipt.get('v1_test_accessed') is not False:
        raise RuntimeError('Qualification reports V1-test access')
    bundle = receipt.get('qualification_bundle') or {}
    if not bundle.get('slug'):
        raise RuntimeError('Qualification PASS did not publish the immutable qualification handoff')
    summary = {
        'status': receipt['status'],
        'materialization_id': receipt['materialization_id'],
        'qualification_science_sha256': receipt['qualification_science_sha256'],
        'preinference_qa_sha256': receipt['preinference_qa_sha256'],
        'qualification_bundle': bundle['slug'],
        'protected_external_prediction_count': receipt['protected_external_prediction_count'],
        'v1_test_accessed': receipt['v1_test_accessed'],
        'next_gate': 'Review the qualification evidence and attach this exact qualification dataset for claim.',
    }
else:
    if receipt.get('status') != 'PASS_TRACKB_CLOSED_PRIVATE_ARCHIVED':
        raise RuntimeError(f"Claim did not close safely: {receipt.get('status')}")
    summary = {
        'status': receipt['status'],
        'materialization_id': receipt['materialization_id'],
        'qualification_recomputed': receipt.get('qualification_recomputed'),
        'trackb_science_sha256': receipt['trackb_science_sha256'],
        'closure_sha256': receipt['closure_sha256'],
        'final_qa_sha256': receipt['final_qa_sha256'],
        'private_evidence': receipt['private_evidence']['slug'],
        'public_evidence_zip': receipt['public_evidence_zip'],
        'next_gate': 'Archive the public-safe evidence package; do not rerun science.',
    }

print(json.dumps(summary, indent=2, sort_keys=True))
